In [0]:
# Read Dataset and create Spark DataFrame
base_volume = "/Volumes/dbx_apps_poc/mlpractice/volumes/mlmodel/diabetes_data"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{base_volume}/diabetes.csv")
display(df.limit(5))

In [0]:
# Split train-test data 
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

print(f"Training dataset count: {train_df.count()}")
print(f"Test dataset count: {test_df.count()}")

In [0]:
from pyspark.ml.classification import LogisticRegression

from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

import mlflow

# -----------------------------------
# 1. Feature engineering
# -----------------------------------
feature_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

scaler = MinMaxScaler(
    inputCol="features_raw",
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="Outcome")

pipeline = Pipeline(stages=[assembler, scaler, lr])

# -----------------------------------
# 2. Param Grid
# -----------------------------------
paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.001, 0.01, 0.1, 1.0])   # C equivalent
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])   # L2, mix, L1
    .addGrid(lr.maxIter, [50, 100, 200])
    .build()
)

# -----------------------------------
# 3. Evaluator
# -----------------------------------
evaluator =  MulticlassClassificationEvaluator(labelCol="Outcome", predictionCol='prediction', metricName= 'accuracy')
      

# -----------------------------------
# 4. CrossValidator
# -----------------------------------
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4   # important in Fabric
)

# -----------------------------------
# 5. MLflow tracking (Fabric auto-integrated)
# -----------------------------------
mlflow.set_experiment("/Users/abay-molla.kassa@capgemini.com/fabric_spark_logreg")

with mlflow.start_run():

    model = cv.fit(train_df)

    best_model = model.bestModel

    # Log best params manually
    best_lr = best_model.stages[-1] if hasattr(best_model, "stages") else best_model

    mlflow.log_param("regParam", best_lr._java_obj.getRegParam())
    mlflow.log_param("elasticNetParam", best_lr._java_obj.getElasticNetParam())
    mlflow.log_param("maxIter", best_lr._java_obj.getMaxIter())

    # Evaluate
    preds = best_model.transform(test_df)
    auc = evaluator.evaluate(preds)

    accuracy_score = evaluator.evaluate(preds)

    mlflow.log_metric("AUC", auc)

    print("Best AUC:", auc)

    """# Weighted (overall metics)
    overallPrecision = evaluator.evaluate(preds, {evaluator.metricName: 'areaUnderPR'})
    print('\nOverall metrics:', overallPrecision)

    overallRecall = evaluator.evaluate(preds, {evaluator.metricName: 'weightedRecall'})
    print('\tRecall:', overallRecall)

    overallF1 = evaluator.evaluate(preds, {evaluator.metricName: 'weightedFMeasure'})
    print('\tF1 Score:', overallF1)"""
    

In [0]:
preds_pd = preds.toPandas()[["Outcome", "probability"]]
preds_pd["prob"] = preds_pd["probability"].apply(lambda x: x[1])
#preds_pd = preds_pd.drop(["probability"], axis=1

print(preds_pd)

In [0]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(preds_pd["Outcome"], preds_pd["prob"])
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()
